In [ ]:
import pandas as pd
import requests
import zipfile
import io

# 1. Download the real-world banking dataset
print("Downloading dataset....")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

# Download the ZIP file
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# This data is seperated by semicolon (;), not comma, so we tell pandas to use sep=';'
# Read the 'bank.csv' file from the ZIP (assuming this is the main data file)
with zip_file.open('bank.csv') as f:
	df = pd.read_csv(f, sep=';')

# 2. Save a local copy! (This create a real CSV file in your folder)
df.to_csv("bank_marketing_local.csv", index=False)
print("Saved a local copy to your folder as 'bank_marketing_local.csv' ")

# 3. View the first 5 rows of the dataset               
display(df.head())


In [ ]:
# Load the local CSV file into a dataframe
df_bank_marketing_local = pd.read_csv("bank_marketing_local.csv")   

print("===1. HEAD(): Do the colums make sense?===") 
display(df_bank_marketing_local.head())

print("\n===2. INFO(): What are the datatypes? Are there any missing values?===")
display(df_bank_marketing_local.info())

print("\n===3. DESCRIBE(): What are the summary statistics?===")
display(df_bank_marketing_local.describe())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. ANswer the Boss's question mathematically
# We group by the final column'Y' (which is 'yes' or 'no') and find the average age
age_analysis = df.groupby('y')['age'].mean().reset_index()

print("----AVERAGE AGE OF CUSTOMERS ----")
display(age_analysis)

# 2. Answer the Boss's Question visually
plt.figure(figsize=(6, 4))

# A boxplot is perfect for this. It shows the distribution of ages for both groups.
sns.boxplot(data=df, x='y', y='age', palette=['#ff9999', '#66b3ff'])

plt.title("Age Distribution: Who says yes to the bank?", fontweight='bold')
plt.xlabel("DId they open an account? (y)", fontsize=12)
plt.ylabel("Age of Customer", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Turn 'yes' and 'no' into 1 and 0
df['y_numeric'] = df['y'].map({'yes': 1, 'no': 0})

# 2. Calculate the Success Rate by Job
# If 10 out of 100 students say 'yes', the mean() will be 0.10 (10%)
job_analysis = df.groupby('job')['y_numeric'].mean().reset_index()

# Multiply by 100 to make it a percentage, and sort it from hoghest to lowest
job_analysis['Success_Rate (%)'] = job_analysis['y_numeric'] * 100
job_analysis = job_analysis.sort_values(by='Success_Rate (%)', ascending=False) 

print("----SUCCESS RATE BY JOB TYPE----")
display(job_analysis[['job', 'Success_Rate (%)']])

# 3. Visualize the Strategy for the Boss
plt.figure(figsize=(10, 6))
sns.barplot(data=job_analysis, x='Success_Rate (%)', y='job', palette='viridis')

plt.title("Bank Marketing Success Rate by Job", fontweight='bold', fontsize=14)
plt.xlabel("Percentage who said YES (%)", fontsize=12)
plt.ylabel("Job Category", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare the data for modeling
print("---PREPARING DATA FOR AI---")
# 1. Select the features we want the AI to study
# We will use 'age', 'balance' (How much money they have), 'job', and 'marital status'ArithmeticError
features = ['age', 'balance', 'job', 'marital']
x_raw = df[features]
y = df['y_numeric']  # This is our target variable (1 for yes, 0 for no)

# 2. ONE - HOT ENCODING (The magic Trick)
# Watch Pandas turn text into binary numbers (0s and 1s) automatically!
x_encoded = pd.get_dummies(x_raw)

print("How the AI sees the data now:")
display(x_encoded.head(3)) # Look at the new columns!

# 2. THE MACHINE LEARNING PIPELINE
# 3. Split the data (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(x_encoded, y, test_size=0.2, random_state=42)

# 4. Build and Train the Brain (AI)
model = RandomForestClassifier(n_estimators=100, random_state=42)  # A powerful AI model
print("\nTraining the Random Forest AI model...")
model.fit(x_train, y_train)

# 5. Make predictions on the hidden 20% of the data
predictions = model.predict(x_test)

# 6. Grade the AI's performance
accuracy = accuracy_score(y_test, predictions)
print(f"\n AI ACCURACY SCORE: {accuracy * 100:.2f}%")



In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Prove the Trap: What percentage of people actually said 'No' originally?
print("--- THE LAZY AI TEST ---")
lazy_score = df['y_numeric'].value_counts(normalize=True) * 100
print(lazy_score)

# 2. Build the Confusion Matrix
cm = confusion_matrix(y_test, predictions)

# 3. Draw it beautifully
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap= 'Reds',
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])

plt.title("The AI's real Report Cart (Confusion Matrix)", fontweight='bold', fontsize = 12)
plt.show()

In [ ]:
# 1. Build a new Brain, but this time we add class_weight = 'balanced'
# We also limit the depth of the tres (max_depth) so it doesn't overthink
smart_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', max_depth=5)

print("Training the Balanced AI....")
smart_model.fit(x_train, y_train)

# 2. Make new predictions
smart_predictions = smart_model.predict(x_test)

# 3. Build the new confusion matrix
cm_smart = confusion_matrix(y_test, smart_predictions)  

# 4. Draw the new chart side by side with the old one (optional), but let's just draw the new one.
plt.figure(figsize=(6, 5))
sns.heatmap(cm_smart, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])

plt.title("The Balanced AI's Report Card (Confusion Matrix)", fontweight='bold', fontsize=12)
plt.show()

# Let's print the detailed classification report to see precision, recall, and F1-score
print("\n--- THE DETAILED REPORT CARD ---")
print(classification_report(y_test, smart_predictions, target_names=['No', 'Yes']))